In [3]:
!pip install kafka-python

In [4]:
import json
import pandas as pd
from kafka import KafkaProducer

In [5]:
# -----------------------------
# Kafka Producer (batch-friendly)
# -----------------------------
producer = KafkaProducer(
bootstrap_servers="kafka:29092",
value_serializer=lambda v: json.dumps(v).encode("utf-8"),
linger_ms=5000 # batch messages before sending
)

In [6]:
df = pd.read_csv('/home/jovyan/data/olist_order_payments_dataset.csv')
print("Rows to publish:", len(df))

Rows to publish: 103886


In [7]:
df.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [8]:
# ==============================
# CEK DATA SEBELUM TRANSFORMASI
# ==============================

# 1. Ukuran data
print("=== Shape ===")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

# 2. Tipe data
print("\n=== Data Types ===")
print(df.dtypes)

# 3. Jumlah null per kolom
print("\n=== Null Count ===")
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_summary = pd.DataFrame({"null_count": null_counts, "null_%": null_pct})
print(null_summary[null_summary["null_count"] > 0])  # hanya tampilkan yang ada null

# 4. Duplikat
print("\n=== Duplicates ===")
print(f"Duplicate rows     : {df.duplicated().sum():,}")
print(f"Duplicate order_id : {df['order_id'].duplicated().sum():,}")

# 5. Nilai unik kolom kategorikal
print("\n=== Unique Values - payment_type ===")
print(df["payment_type"].value_counts())

# 7. Preview data
print("=== Preview (head 3) ===")
df.head(3)

=== Shape ===
Rows: 103,886 | Columns: 5

=== Data Types ===
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

=== Null Count ===
Empty DataFrame
Columns: [null_count, null_%]
Index: []

=== Duplicates ===
Duplicate rows     : 0
Duplicate order_id : 4,446

=== Unique Values - payment_type ===
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64
=== Preview (head 3) ===


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


In [9]:
# Tidak ada transformasi yang diperlukan
# Semua tipe data sudah sesuai

print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum())
print("\nData siap dipublish ke Kafka")

order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

Null counts:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

Data siap dipublish ke Kafka


In [10]:
# -----------------------------
# Publish to Kafka in batch
# -----------------------------
for _, row in df.iterrows():
    msg = {
        "order_id"            : row["order_id"],
        "payment_sequential"  : int(row["payment_sequential"]),
        "payment_type"        : row["payment_type"],   # bisa None jika 'not_defined'
        "payment_installments": int(row["payment_installments"]),
        "payment_value"       : float(row["payment_value"]),
    }
    producer.send("payments", msg)

producer.flush()
print("Batch publish to Kafka finished")

Batch publish to Kafka finished
